In [41]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
from pomelo.utils import Hal

In [42]:
end_date = date.today().strftime("%Y-%m-%d")
six_months = date.today() - relativedelta(months=+6)
start_date = six_months.strftime("%Y-%m-%d")

In [43]:
end_date

'2021-07-28'

In [44]:
start_date

'2021-01-28'

In [91]:
hal = Hal()

sql = f"""
SELECT dp.id_product_attribute,
        CASE WHEN fs.id_shop = 5 THEN 'ID' 
          WHEN fs.id_shop = 2 THEN 'MY'
          WHEN fs.id_shop = 11 THEN 'MY'
          WHEN fs.id_shop = 14 THEN 'MY'
          ELSE 'TH' END as warehouse
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=0 AND DATEDIFF(day,dp.date_released,fs.order_date)<=6 THEN fs.net_units_sold ELSE 0 END) as week1
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=7 AND DATEDIFF(day,dp.date_released,fs.order_date)<=13 THEN fs.net_units_sold ELSE 0 END) as week2
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=14 AND DATEDIFF(day,dp.date_released,fs.order_date)<=20 THEN fs.net_units_sold ELSE 0 END) as week3
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=21 AND DATEDIFF(day,dp.date_released,fs.order_date)<=27 THEN fs.net_units_sold ELSE 0 END) as week4
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=28 AND DATEDIFF(day,dp.date_released,fs.order_date)<=34 THEN fs.net_units_sold ELSE 0 END) as week5
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=35 AND DATEDIFF(day,dp.date_released,fs.order_date)<=41 THEN fs.net_units_sold ELSE 0 END) as week6
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=42 AND DATEDIFF(day,dp.date_released,fs.order_date)<=48 THEN fs.net_units_sold ELSE 0 END) as week7
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=49 AND DATEDIFF(day,dp.date_released,fs.order_date)<=55 THEN fs.net_units_sold ELSE 0 END) as week8
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=56 AND DATEDIFF(day,dp.date_released,fs.order_date)<=62 THEN fs.net_units_sold ELSE 0 END) as week9
    --,SUM(fs.net_units_sold) as tot_net_units_sold
    ,SUM(CASE WHEN DATEDIFF(day,dp.date_released,fs.order_date)>=0 AND DATEDIFF(day,dp.date_released,fs.order_date)<=62 THEN fs.net_units_sold ELSE 0 END) as tot_net_units_sold
FROM dwh.fact_sales fs
LEFT JOIN dwh.dim_product dp
ON fs.id_product_attribute  = dp.id_product_attribute AND fs.id_product = dp.id_product 
WHERE dp.date_released BETWEEN '{start_date}' AND '{end_date}'
AND dp.parent_product_line NOT IN ('Free Gift')
AND dp.product_line NOT IN('Bags','3P Apparel','Shoes')
AND dp.henry_category_1 NOT IN ('Accessories','Bags','Bath&Body','Beverage Container','Cosmetics','Hair','Miscellaneous','Shoes','Skin Care','Stationery')
AND dp.product_cost_usd_first_order_date > 0 
AND dp.brand IN ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio','Blackdog BKK')
AND dp.release_collection_name IS NOT NULL
AND fs.order_type IN ('Marketplace','Web','Android','iOS','Partner','Site to Store')
Group by 1,2

"""

In [92]:
week_dist = hal.get_pandas_df(sql)

In [93]:
week_dist

,id_product_attribute,warehouse,week1,week2,week3,week4,week5,week6,week7,week8,week9,tot_net_units_sold
0,137126,TH,5,1,0,0,0,0,1,3,1,11
1,140051,TH,12,2,1,0,0,0,0,0,0,15
2,129710,TH,4,0,3,5,2,1,2,0,0,17
3,138522,MY,0,0,0,0,0,0,1,2,0,3
4,132540,TH,12,11,2,1,3,7,5,0,0,41
...,...,...,...,...,...,...,...,...,...,...,...,...
11860,129833,ID,0,0,0,0,0,0,0,0,0,0
11861,119098,ID,0,0,0,0,0,0,1,0,0,1
11862,132930,ID,0,0,0,0,0,0,0,0,0,0
11863,135324,ID,0,0,1,0,0,0,0,0,0,1


In [94]:
week_dist = week_dist.groupby("warehouse").sum().reset_index()
del week_dist["id_product_attribute"]
week_dist

,warehouse,week1,week2,week3,week4,week5,week6,week7,week8,week9,tot_net_units_sold
0,ID,835,1313,1322,1534,1110,1377,1094,1022,1246,10853
1,MY,3215,3121,2995,2896,2192,2337,2143,1826,1932,22657
2,TH,15381,9647,6442,6083,5115,5258,5955,4420,4210,62511


In [95]:
week_dist2 = week_dist.iloc[:, :10]

In [96]:
week_dist_unpivot = week_dist2.melt(
    id_vars=["warehouse"], var_name="week_id", value_name="net_units_sold"
)
week_dist_unpivot

,warehouse,week_id,net_units_sold
0,ID,week1,835
1,MY,week1,3215
2,TH,week1,15381
3,ID,week2,1313
4,MY,week2,3121
5,TH,week2,9647
6,ID,week3,1322
7,MY,week3,2995
8,TH,week3,6442
9,ID,week4,1534


In [97]:
df = week_dist_unpivot.merge(
    week_dist[["warehouse", "tot_net_units_sold"]], on="warehouse"
)
df

,warehouse,week_id,net_units_sold,tot_net_units_sold
0,ID,week1,835,10853
1,ID,week2,1313,10853
2,ID,week3,1322,10853
3,ID,week4,1534,10853
4,ID,week5,1110,10853
5,ID,week6,1377,10853
6,ID,week7,1094,10853
7,ID,week8,1022,10853
8,ID,week9,1246,10853
9,MY,week1,3215,22657


In [98]:
df["week_dist"] = df["net_units_sold"] / df["tot_net_units_sold"]

In [99]:
df

,warehouse,week_id,net_units_sold,tot_net_units_sold,week_dist
0,ID,week1,835,10853,0.076937
1,ID,week2,1313,10853,0.120980
2,ID,week3,1322,10853,0.121810
3,ID,week4,1534,10853,0.141343
4,ID,week5,1110,10853,0.102276
5,ID,week6,1377,10853,0.126877
6,ID,week7,1094,10853,0.100802
7,ID,week8,1022,10853,0.094168
8,ID,week9,1246,10853,0.114807
9,MY,week1,3215,22657,0.141899


In [100]:
df.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/week_dist.csv",
    index=False,
)

In [101]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    subplot_titles=("TH", "MY", "ID"),
)

fig.add_trace(
    go.Scatter(
        x=df[df["warehouse"] == "ID"]["week_id"],
        y=df[df["warehouse"] == "ID"]["week_dist"],
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df[df["warehouse"] == "MY"]["week_id"],
        y=df[df["warehouse"] == "MY"]["week_dist"],
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df[df["warehouse"] == "TH"]["week_id"],
        y=df[df["warehouse"] == "TH"]["week_dist"],
    ),
    row=1,
    col=1,
)

fig.update_layout(height=900, width=900, title_text="week dist by warehouse")

fig.write_html(
    f"/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/online_dfm_v2/plot_week_dist.html"
)